In [23]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

##################################
# GARCH-X，照計畫書的公式，但效果很差， VAR的 STD 是用 GARCH的SHAPE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [24]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"


Agent pid 75446
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [26]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [27]:
# =============================================================================
# ES1 GK-LSTM VaR-t  ── B 版本 (Walk-Forward Warm Update)
# =============================================================================
# 研究設計：
#   Stage 1 ── Base Model 訓練（2006-2021, batch training）
#   Stage 2 ── Walk-Forward 評估（2022-2025）
#               每日：predict → record → warm update → slide
#
# 目標變數 Y : gk_vol_daily  (GK 單日波動度，σ 尺度，不需再開根號)
# 輸入特徵 X : ES1_LN_RET, gk_vol_daily, garch_vol, VIX_CLOSE  (lookback=20)
# VaR      : t 分配, μ=0, shape clip(6,10), α=0.05 / 0.01
# Backtest : Kupiec UC + Christoffersen CC  (訓練期 & 滾動評估期)
# Baseline : GARCH 模型 (來自 es1_volatility_all_methods.csv)
# =============================================================================

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import t as tdist, chi2, pearsonr

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
warnings.filterwarnings('ignore')

# =============================================================================
# 0.  設定區  ── 所有參數集中在此，修改這裡即可
# =============================================================================

# ── 資料路徑 ──────────────────────────────────────────────────────────────────
PATH_DAY  = "./filtered_output/df_day.csv"
PATH_VOL  = "./real_volatility_multi_var/es1_volatility_all_methods.csv"
PATH_GK_DAILY  = "./real_volatility_multi_var/gk_vol_daily_var_es_daily.csv"

# ── 樣本切分 ──────────────────────────────────────────────────────────────────
TRAIN_START = "2006-01-01"
TRAIN_END   = "2021-12-31"
TEST_START  = "2022-01-01"


# ── 特徵 & 目標 ───────────────────────────────────────────────────────────────
# [改動1] TARGET_COL 由 gk_vol_daily → gk_daily_VaR_price_95
FEATURE_COLS = ['garch_vol', 'ES1_LOW', 'gk_daily_VaR_price_95']
TARGET_COL   = 'gk_daily_VaR_price_95'


OUTPUT_DIR = "./LSTM_B_diagnostics"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── GPU 設定（有 GPU 自動加速；無 GPU 正常跑 CPU）────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("ℹ  No GPU, running on CPU")


# =============================================================================
# 1.  讀資料 & 建母資料表
# =============================================================================

def load_master(path_day: str, path_vol: str, path_gk_daily: str) -> pd.DataFrame:
    # """
    # 合併 df_day.csv 與 es1_volatility_all_methods.csv，
    # 以 DATE 對齊，篩選 2006 年之後，刪除特徵缺值。
    # """
    # ── df_day ──
    day = pd.read_csv(path_day)
    if 'DATE' not in day.columns:
        day = day.reset_index().rename(columns={day.columns[0]: 'DATE'})
    day['DATE'] = pd.to_datetime(day['DATE'], errors='coerce').dt.normalize()

    keep_day = ['DATE', 'ES1_LN_RET', 'ES1_CLOSE', 'ES1_LOW','ES1_VOLUME',
                'VIX_LN_RET', 'VIX_CLOSE']
    day = day[[c for c in keep_day if c in day.columns]].dropna(subset=['DATE'])

    # ── vol / garch / shape ──
    vol = pd.read_csv(path_vol)
    if 'DATE' not in vol.columns:
        vol = vol.reset_index().rename(columns={vol.columns[0]: 'DATE'})
    vol['DATE'] = pd.to_datetime(vol['DATE'], errors='coerce').dt.normalize()

    keep_vol = ['DATE', 'gk_vol_daily', 'garch_vol', 'shape']
    vol = vol[[c for c in keep_vol if c in vol.columns]].dropna(subset=['DATE'])

    # ── gk va95 ret / gk va95 price ──
    gk = pd.read_csv(path_gk_daily)
    if 'DATE' not in gk.columns:
        gk = gk.reset_index().rename(columns={gk.columns[0]: 'DATE'})
    gk['DATE'] = pd.to_datetime(gk['DATE'], errors='coerce').dt.normalize()

    keep_gk = ['DATE', 'VaR_ret_95', 'VaR_price_95']
    gk = gk[[c for c in keep_gk if c in gk.columns]].dropna(subset=['DATE'])
    gk = gk.rename(columns={
    'VaR_ret_95': 'gk_daily_VaR_ret_95',
    'VaR_price_95': 'gk_daily_VaR_price_95'
    })


    # ── merge ──
    df = pd.merge(day, vol, on='DATE', how='inner')
    df = pd.merge(df, gk, on='DATE', how='inner')

    df = df.sort_values('DATE').reset_index(drop=True)
    df = df[df['DATE'] >= TRAIN_START].reset_index(drop=True)
    df = df.dropna(subset=FEATURE_COLS + ['shape']).reset_index(drop=True)

    print(f"Master table: {len(df):,} rows  "
          f"({df['DATE'].min().date()} ~ {df['DATE'].max().date()})")
    return df


df = load_master(PATH_DAY, PATH_VOL, PATH_GK_DAILY)

for c in df.columns:
    print(c)


df_data = pd.DataFrame(df)

df_data_path = os.path.join(OUTPUT_DIR, "df_data.csv")
df_data.to_csv(df_data_path, index=False, encoding='utf-8-sig')

print(f"✓ Saved: {df_data_path}")


✓ GPU: 1 device(s)
Master table: 4,922 rows  (2006-01-03 ~ 2025-06-30)
DATE
ES1_LN_RET
ES1_CLOSE
ES1_LOW
ES1_VOLUME
VIX_LN_RET
VIX_CLOSE
gk_vol_daily
garch_vol
shape
gk_daily_VaR_ret_95
gk_daily_VaR_price_95
✓ Saved: ./LSTM_B_diagnostics/df_data.csv


In [31]:
# =============================================================================
# ES1 GK-LSTM — 直接預測 gk_daily_VaR_price_95  （B 版本 Walk-Forward）
# =============================================================================
# 研究設計：
#   Stage 1 ── Base Model 訓練（2006-2021, batch training）
#   Stage 2 ── Walk-Forward 評估（2022-2025）
#               每日：predict → record → warm update → slide
#
# 目標變數 Y : gk_daily_VaR_price_95  (價格層 VaR，直接預測)
# 輸入特徵 X : gk_vol_daily(log), garch_vol, VIX_CLOSE, ES1_LN_RET
# 三方比較   : GK Realized / GARCH / LSTM_GK  （訓練期 & 測試期）
# 隱含波動   : 由預測 VaR_price 反推 sigma_implied
# Backtest   : Kupiec UC + Christoffersen CC
#
# ── 相較於舊版 sp500_3 的主要改動 ──────────────────────────────────────────────
# [1] TARGET_COL: gk_vol_daily → gk_daily_VaR_price_95
# [2] 特徵縮放: X 仍用 log(gk_vol_daily)；Y 改用 log(-gk_daily_VaR_price_95/ES1_CLOSE_prev)
#     (因為 VaR_price 是正數且數值大，取 log 比例讓模型更容易學習；反縮放時還原)
# [3] 新增：sigma_implied 反推（由預測 VaR_price 逆推隱含波動）
# [4] 新增：三模型 VaR_price 比較圖（GK / GARCH / LSTM_GK）
# [5] 新增：隱含波動 vs gk_vol_daily vs garch_vol 比較圖
# [6] 新增：按 quintile 的 bias 診斷圖（VaR_price 版本）
# [7] VaR 回測邏輯改為 price-level violation（ES1_CLOSE < VaR_price_pred）
# [8] GARCH VaR_price 用相同公式建構，確保三方可比
# [9] 新增：描述統計 CSV + 隱含波動統計 CSV
# [10]輸出 df_data.csv 沿用既有母資料表路徑
# =============================================================================



# =============================================================================
# 0.  設定區  ── 所有參數集中在此
# =============================================================================

# ── 資料路徑 ──────────────────────────────────────────────────────────────────
# 直接使用已整合好的母資料表 df_data.csv（三份來源已完成 inner merge）
PATH_MASTER = "./LSTM_B_diagnostics/df_data.csv"   # ← 你的母資料表路徑

LOOKBACK = 20

# ── Base Model 超參數 ─────────────────────────────────────────────────────────
LSTM_UNITS  = 64
DROPOUT     = 0.2
LR_BASE     = 1e-3
EPOCHS      = 100
BATCH_SIZE  = 32
PATIENCE_ES = 20
PATIENCE_LR = 10

# ── Walk-Forward Warm Update 超參數 ──────────────────────────────────────────
LR_WARM     = 1e-4         # 比 Base LR 低 10×
WARM_EPOCHS = 1            # 每次 1 epoch，避免單日噪音過擬合
WARM_BATCH  = 1

# ── VaR 設定 ──────────────────────────────────────────────────────────────────
ALPHA_95 = 0.05
ALPHA_99 = 0.01
NU_CLIP  = (6, 10)         # shape 截尾範圍

# ── GPU ───────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("ℹ  No GPU, running on CPU")


# =============================================================================
# 1.  讀母資料表
# =============================================================================

# df = pd.read_csv(PATH_MASTER, parse_dates=['DATE'])
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce').dt.normalize()
df = df.sort_values('DATE').reset_index(drop=True)
df = df[(df['DATE'] >= TRAIN_START)].reset_index(drop=True)
# df = df.dropna(subset=FEATURE_COLS + [TARGET_COL, 'shape', 'ES1_CLOSE']).reset_index(drop=True)

print(f"Master table: {len(df):,} rows  "
      f"({df['DATE'].min().date()} ~ {df['DATE'].max().date()})")
print(f"Columns: {list(df.columns)}")

# 描述統計表（需求書第十三節）
desc_cols = ['gk_vol_daily','garch_vol','gk_daily_VaR_ret_95',
             'gk_daily_VaR_price_95','ES1_LN_RET','VIX_CLOSE']
df_desc = df[desc_cols].describe().T.round(6)
df_desc.to_csv(os.path.join(OUTPUT_DIR, "desc_stats.csv"), encoding='utf-8-sig')
print("✓ Saved desc_stats.csv")


# =============================================================================
# 2.  特徵縮放 & 滑動視窗序列建構
# =============================================================================

train_mask = (df['DATE'] >= TRAIN_START) & (df['DATE'] <= TRAIN_END)

###################################################################################
# 有爭議
# ── X 縮放 ────────────────────────────────────────────────────────────────────
# [改動2a] gk_vol_daily 取 log（降低右偏、減少 variance suppression）
#          其餘特徵保持 MinMaxScaler
X_raw = df[FEATURE_COLS].values.astype(np.float32)

low_idx = FEATURE_COLS.index('ES1_LOW')
var_idx = FEATURE_COLS.index('gk_daily_VaR_price_95')

X_raw[:, low_idx] = np.log(np.clip(X_raw[:, low_idx], 1e-8, None))
X_raw[:, var_idx] = np.log(np.clip(X_raw[:, var_idx], 1e-8, None))

X_scaled_all = X_raw

############ 做 minmax 先拿到
# scaler_X = MinMaxScaler()
# scaler_X.fit(X_raw[train_mask])
# X_scaled_all = scaler_X.transform(X_raw)

# ── Y 縮放 ────────────────────────────────────────────────────────────────────
# [改動2b] Y = gk_daily_VaR_price_95 (正值，單位為指數點)
# 取 log 讓模型預測「對數 VaR price」，反縮放時 exp() 還原
# 這樣可大幅降低量綱問題，且確保預測值恆正
Y_raw_all = df[TARGET_COL].values.astype(np.float64)
Y_log_all = np.log(np.clip(Y_raw_all, 1.0, None))   # log(VaR_price)
Y_scaled_all = Y_log_all


############ 做 minmax 先拿到
# scaler_y = MinMaxScaler()
# scaler_y.fit(Y_log_all[train_mask].reshape(-1, 1))
# Y_scaled_all = scaler_y.transform(Y_log_all.reshape(-1, 1)).ravel().astype(np.float32)

def descale_var_price(y_scaled: np.ndarray) -> np.ndarray:
    # """反縮放：scaled → log_var_price → var_price（正值）"""
    # y_log = scaler_y.inverse_transform(
    #     np.array(y_scaled, dtype=np.float64).reshape(-1, 1)).ravel()
    # return np.exp(y_log)

    return np.exp(np.array(y_scaled, dtype=np.float64))

#############################################################################################################

# ── 滑動視窗序列建構 ──────────────────────────────────────────────────────────
def build_sequences(X_sc, Y_sc, dates, shapes, returns, closes, true_y_raw, lookback):
    # """
    # X[i] = X[i-lookback:i]  (前 lookback 天特徵)
    # Y[i] = Y[i]              (第 i 天目標值，已縮放)
    # """
    Xs, Ys = [], []
    s_dates, s_shapes, s_ret, s_close, s_true = [], [], [], [], []

    for i in range(lookback, len(X_sc)):
        Xs.append(X_sc[i - lookback: i])
        Ys.append(Y_sc[i])
        s_dates.append(dates[i])
        s_shapes.append(shapes[i])
        s_ret.append(returns[i])
        s_close.append(closes[i])
        s_true.append(true_y_raw[i])

    return (np.array(Xs, dtype=np.float32),
            np.array(Ys, dtype=np.float32),
            np.array(s_dates),
            np.array(s_shapes, dtype=np.float32),
            np.array(s_ret,    dtype=np.float32),
            np.array(s_close,  dtype=np.float32),
            np.array(s_true,   dtype=np.float32))


(X_seq, Y_seq, seq_dates, seq_shapes,
 seq_ret, seq_close, seq_true) = build_sequences(
    X_scaled_all, Y_scaled_all,
    df['DATE'].values, df['shape'].values,
    df['ES1_LN_RET'].values, df['ES1_CLOSE'].values,
    Y_raw_all,LOOKBACK
)

# ── 訓練 / 測試切分 ───────────────────────────────────────────────────────────
tr_mask = pd.to_datetime(seq_dates) <= pd.Timestamp(TRAIN_END)
te_mask = pd.to_datetime(seq_dates) >= pd.Timestamp(TEST_START)

X_train, Y_train    = X_seq[tr_mask], Y_seq[tr_mask]
X_test,  Y_test     = X_seq[te_mask], Y_seq[te_mask]
dates_tr, dates_te  = pd.to_datetime(seq_dates[tr_mask]), pd.to_datetime(seq_dates[te_mask])
shape_tr, shape_te  = seq_shapes[tr_mask], seq_shapes[te_mask]
ret_tr,   ret_te    = seq_ret[tr_mask],    seq_ret[te_mask]
close_tr, close_te  = seq_close[tr_mask],  seq_close[te_mask]
true_y_tr, true_y_te = seq_true[tr_mask],  seq_true[te_mask]  # gk_daily_VaR_price_95

n_features = X_train.shape[2]
print(f"\nTrain: {X_train.shape}  ({dates_tr[0].date()} ~ {dates_tr[-1].date()})")
print(f"Test : {X_test.shape}   ({dates_te[0].date()} ~ {dates_te[-1].date()})")
print(f"n_features = {n_features}")


# =============================================================================
# 3.  LSTM 模型建構（tensorflow.keras）
# =============================================================================

def build_lstm_model(lookback, n_feat, lstm_units=LSTM_UNITS,
                     dropout=DROPOUT, lr=LR_BASE):
    # """
    # Input → LSTM → Dropout → Dense(linear)
    # 輸出為 log-scaled VaR_price，線性輸出層（無 softplus）
    # 因為 Y 已 log-scaled，線性輸出足以確保反縮放後為正值
    # 損失：Huber（對尾端 VaR 極端值更穩健）
    # """
    inp = layers.Input(shape=(lookback, n_feat), name="seq_input")
    x   = layers.LSTM(lstm_units, return_sequences=False, name="lstm_1")(inp)
    x   = layers.Dropout(dropout, name="dropout")(x)
    out = layers.Dense(1, activation='linear', name="var_price_output")(x)

    m = models.Model(inp, out, name="GK_LSTM_VaR_Price")
    m.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.Huber(delta=1.0)  # Huber 對極端 VaR 值更穩健
    )
    return m


# =============================================================================
# 4.  Stage 1：Base Model 訓練（2006–2021）
# =============================================================================

print("\n" + "="*60)
print("STAGE 1: Base Model Training (2006-2021)")
print("="*60)

base_model = build_lstm_model(LOOKBACK, n_features, lr=LR_BASE)
base_model.summary()

cb_list = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE_ES,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=PATIENCE_LR, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "base_model_best.keras"),
        monitor='val_loss', save_best_only=True, verbose=0),
]

history = base_model.fit(
    X_train, Y_train,
    validation_split=0.1,   # 末 10% 作驗證集（時序保留）
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,           # 時間序列嚴禁打亂
    callbacks=cb_list,
    verbose=1
)
print(f"✓ Base model trained: {len(history.history['loss'])} epochs")

# ── 訓練期 in-sample 預測 ────────────────────────────────────────────────────
pred_tr_s = base_model.predict(X_train, verbose=0).ravel()
pred_tr   = descale_var_price(pred_tr_s)    # gk_daily_VaR_price_95 預測值（正數）

rmse_tr = float(np.sqrt(mean_squared_error(true_y_tr, pred_tr)))
mae_tr  = float(mean_absolute_error(true_y_tr, pred_tr))
corr_tr = float(pearsonr(pred_tr, true_y_tr)[0])
print(f"[Train] RMSE={rmse_tr:.2f}  MAE={mae_tr:.2f}  Corr={corr_tr:.4f}")

pred_tr_log = pred_tr_s
true_tr_log = Y_train

print("MAE log-scale:", np.mean(np.abs(pred_tr_log - true_tr_log)))
print("RMSE log-scale:", np.sqrt(np.mean((pred_tr_log - true_tr_log)**2)))

dates_plot = pd.to_datetime(dates_tr)
true_log = np.array(Y_train).ravel()
pred_log = np.array(pred_tr_s).ravel()

print(len(dates_plot), len(true_log), len(pred_log))

check_plot_path = os.path.join(OUTPUT_DIR, "train_log_check.png")

plt.figure(figsize=(14,5))
plt.plot(dates_plot, true_log, label='True log VaR', linewidth=1)
plt.plot(dates_plot, pred_log, label='Pred log VaR', linewidth=1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(check_plot_path, dpi=200, bbox_inches='tight')
plt.close()

print(f"✓ Saved: {check_plot_path}")

✓ GPU: 1 device(s)
Master table: 4,922 rows  (2006-01-03 ~ 2025-06-30)
Columns: ['DATE', 'ES1_LN_RET', 'ES1_CLOSE', 'ES1_LOW', 'ES1_VOLUME', 'VIX_LN_RET', 'VIX_CLOSE', 'gk_vol_daily', 'garch_vol', 'shape', 'gk_daily_VaR_ret_95', 'gk_daily_VaR_price_95']
✓ Saved desc_stats.csv

Train: (4025, 20, 3)  (2006-01-31 ~ 2021-12-31)
Test : (877, 20, 3)   (2022-01-03 ~ 2025-06-30)
n_features = 3

STAGE 1: Base Model Training (2006-2021)


Model: "GK_LSTM_VaR_Price"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ seq_input (InputLayer)          │ (None, 20, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ var_price_output (Dense)        │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,473 (68.25 KB)

 Trainable params: 17,473 (68.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 1.2278 - val_loss: 0.0537 - learning_rate: 0.0010
Epoch 2/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1857 - val_loss: 0.0385 - learning_rate: 0.0010
Epoch 3/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1788 - val_loss: 0.0584 - learning_rate: 0.0010
Epoch 4/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1784 - val_loss: 0.0490 - learning_rate: 0.0010
Epoch 5/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1683 - val_loss: 0.0585 - learning_rate: 0.0010
Epoch 6/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1645 - val_loss: 0.0536 - learning_rate: 0.0010
Epoch 7/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1526 - val_loss: 0.0701 - learning_rate: 0.0010
Epoch 8/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1480 - val_loss: 0.0285 - learning_rate: 0.0010
Epoch 9/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1499 - val_loss: 0.0903 - learning_rate: 0.0010


In [21]:
# ── Frozen 測試集預測（A版基準） ──────────────────────────────────────────────
pred_frozen_s = base_model.predict(X_test, verbose=0).ravel()
pred_frozen   = descale_var_price(pred_frozen_s)

rmse_frozen = float(np.sqrt(mean_squared_error(true_y_te, pred_frozen)))
mae_frozen  = float(mean_absolute_error(true_y_te, pred_frozen))
corr_frozen = float(pearsonr(pred_frozen, true_y_te)[0])
print(f"[Frozen] RMSE={rmse_frozen:.2f}  MAE={mae_frozen:.2f}  Corr={corr_frozen:.4f}")

base_model.save(os.path.join(OUTPUT_DIR, "base_model_final.keras"))
print("✓ Base model saved.")


# =============================================================================
# 5.  Stage 2：Walk-Forward Warm Update（2022–2025）
# =============================================================================

print("\n" + "="*60)
print(f"STAGE 2: Walk-Forward Warm Update (2022-2025)")
print(f"  Warm LR={LR_WARM}, Epochs={WARM_EPOCHS}, Batch={WARM_BATCH}")
print("="*60)

warm_model = build_lstm_model(LOOKBACK, n_features, lr=LR_WARM)
warm_model.set_weights(base_model.get_weights())

# 驗證權重複製
_cb = base_model.predict(X_test[:3], verbose=0).ravel()
_cw = warm_model.predict(X_test[:3], verbose=0).ravel()
assert np.allclose(_cb, _cw, atol=1e-5), "Weight copy failed!"
print("✓ Warm model weights copied.")

full_dates     = df['DATE'].values
test_pos_start = np.searchsorted(full_dates, np.datetime64(TEST_START))

walk_pred   = []   # LSTM 預測的 gk_daily_VaR_price_95（更新前）
walk_dates  = []
walk_shapes = []
walk_ret    = []
walk_close  = []   # ES1_CLOSE（判斷 price-level violation 用）
walk_true   = []   # 真實 gk_daily_VaR_price_95
walk_garch  = []   # 同期 garch_vol（GARCH baseline 用）
walk_gk_vol = []   # 同期 gk_vol_daily（隱含波動比較用）

n_walk = len(full_dates) - test_pos_start
print(f"Walk-forward: {n_walk} days")

for i, pos in enumerate(range(test_pos_start, len(df))):
    if pos < LOOKBACK:
        continue

    window  = X_scaled_all[pos - LOOKBACK: pos]
    x_input = window.reshape(1, LOOKBACK, n_features).astype(np.float32)

    # Step A&B: 預測 + 記錄（必須在 warm update 前）
    y_pred_s   = warm_model.predict(x_input, verbose=0).ravel()[0]
    y_pred_var = float(descale_var_price(np.array([y_pred_s]))[0])

    walk_pred.append(y_pred_var)
    walk_dates.append(full_dates[pos])
    walk_shapes.append(float(df['shape'].values[pos]))
    walk_ret.append(float(df['ES1_LN_RET'].values[pos]))
    walk_close.append(float(df['ES1_CLOSE'].values[pos]))
    walk_true.append(float(Y_raw_all[pos]))
    walk_garch.append(float(df['garch_vol'].values[pos]))
    walk_gk_vol.append(float(df['gk_vol_daily'].values[pos]))

    # Step C: Warm update（低學習率，1 epoch，不重置權重）
    y_true_s = float(Y_scaled_all[pos])
    warm_model.fit(
        x_input,
        np.array([[y_true_s]], dtype=np.float32),
        epochs=WARM_EPOCHS, batch_size=WARM_BATCH,
        verbose=0, shuffle=False
    )

    if (i + 1) % 200 == 0 or (i + 1) == n_walk:
        print(f"  Walk-forward: {i+1:4d}/{n_walk} done")

# 整理 walk-forward 結果
walk_pred   = np.array(walk_pred,   dtype=np.float64)
walk_dates  = pd.to_datetime(walk_dates)
walk_shapes = np.array(walk_shapes, dtype=np.float64)
walk_ret    = np.array(walk_ret,    dtype=np.float64)
walk_close  = np.array(walk_close,  dtype=np.float64)
walk_true   = np.array(walk_true,   dtype=np.float64)
walk_garch  = np.array(walk_garch,  dtype=np.float64)
walk_gk_vol = np.array(walk_gk_vol, dtype=np.float64)

rmse_wf = float(np.sqrt(mean_squared_error(walk_true, walk_pred)))
mae_wf  = float(mean_absolute_error(walk_true, walk_pred))
corr_wf = float(pearsonr(walk_pred, walk_true)[0])
print(f"\n[Walk-Fwd B] RMSE={rmse_wf:.2f}  MAE={mae_wf:.2f}  Corr={corr_wf:.4f}")
print(f"[Frozen   A] RMSE={rmse_frozen:.2f}  MAE={mae_frozen:.2f}  Corr={corr_frozen:.4f}")

warm_model.save(os.path.join(OUTPUT_DIR, "warm_model_final.keras"))
print("✓ Warm model saved.")


# =============================================================================
# 6.  GARCH & GK Realized VaR_price 建構（三方比較基準）
# =============================================================================

def build_var_t_ret(sigma, shape, alpha=ALPHA_95):
    # """VaR return = sigma × s × t_ν^{-1}(α)  (負值)"""
    nu = np.clip(shape, *NU_CLIP)
    s  = np.sqrt((nu - 2.0) / nu)
    q  = tdist.ppf(alpha, df=nu)
    return sigma * s * q

def var_ret_to_price(close_prev, var_ret):
    # """VaR_price = P_{t-1} × exp(VaR_ret)"""
    return np.array(close_prev, dtype=np.float64) * np.exp(np.array(var_ret, dtype=np.float64))


# ── 訓練期 ────────────────────────────────────────────────────────────────────
# gk_vol_tr = df.loc[tr_mask[LOOKBACK:] if len(tr_mask) > LOOKBACK else tr_mask,
#                    'gk_vol_daily'].values  # 直接用 seq 對應的
# 用 dates_tr 重新從 df 取值（確保對齊）
df_idx = df.set_index('DATE')

gk_vol_tr_arr   = np.array([df_idx.loc[d,'gk_vol_daily']  if d in df_idx.index else np.nan for d in dates_tr])
garch_vol_tr_arr = np.array([df_idx.loc[d,'garch_vol']     if d in df_idx.index else np.nan for d in dates_tr])
shape_tr_arr     = np.array([df_idx.loc[d,'shape']         if d in df_idx.index else np.nan for d in dates_tr])
close_tr_arr     = np.array([df_idx.loc[d,'ES1_CLOSE']     if d in df_idx.index else np.nan for d in dates_tr])
gk_var_ret_tr    = np.array([df_idx.loc[d,'gk_daily_VaR_ret_95']   if d in df_idx.index else np.nan for d in dates_tr])
gk_var_price_tr  = np.array([df_idx.loc[d,'gk_daily_VaR_price_95'] if d in df_idx.index else np.nan for d in dates_tr])

# GARCH VaR (訓練期)：sigma=garch_vol → VaR_ret → VaR_price
# 需要 close_{t-1}：從 df 取 ES1_CLOSE 並 shift
close_all = df.set_index('DATE')['ES1_CLOSE']
def get_prev_close(dates_arr):
    # """取每個日期對應的前一日收盤（來自 df）"""
    all_dates = df['DATE'].values
    result = np.full(len(dates_arr), np.nan)
    for i, d in enumerate(dates_arr):
        pos = np.searchsorted(all_dates, d)
        if pos > 0:
            result[i] = df['ES1_CLOSE'].values[pos - 1]
    return result

# GARCH 這條：有 shift，這是對的，再來算var
prev_close_tr = get_prev_close(dates_tr.values)
garch_var_ret_tr  = build_var_t_ret(garch_vol_tr_arr, shape_tr_arr, ALPHA_95)
garch_var_price_tr = var_ret_to_price(prev_close_tr, garch_var_ret_tr)

# LSTM_GK VaR_price 就是 pred_tr（直接預測）
lstm_var_price_tr = pred_tr

# ── 測試期 ────────────────────────────────────────────────────────────────────
gk_var_price_te  = np.array([df_idx.loc[d,'gk_daily_VaR_price_95'] if d in df_idx.index else np.nan for d in walk_dates])
gk_var_ret_te    = np.array([df_idx.loc[d,'gk_daily_VaR_ret_95']   if d in df_idx.index else np.nan for d in walk_dates])

prev_close_wf = get_prev_close(walk_dates.values)
garch_var_ret_te   = build_var_t_ret(walk_garch, walk_shapes, ALPHA_95)
garch_var_price_te = var_ret_to_price(prev_close_wf, garch_var_ret_te)

lstm_var_price_wf = walk_pred  # walk-forward 的預測（更新前記錄）

# Frozen LSTM VaR_price
# prev_close_frz為了還原
prev_close_frz    = get_prev_close(dates_te.values)
lstm_var_price_frz = pred_frozen


# =============================================================================
# 7.  [新增] 隱含波動反推（sigma_implied）
#     VaR_price = P_{t-1} × exp(VaR_ret)
#     VaR_ret   = sigma × s × q_α
#     → sigma_implied = VaR_ret / (s × q_α)  = abs(log(VaR_price/P_{t-1})) / (s × |q_α|)
# =============================================================================

def implied_sigma(var_price, close_prev, shape, alpha=ALPHA_95):
    # """
    # 由預測 VaR_price 反推隱含波動 sigma_implied。
    # VaR_ret = log(VaR_price / P_{t-1})   (負值)
    # sigma   = |VaR_ret| / (s × |q_α|)
    # """
    nu   = np.clip(np.array(shape, dtype=np.float64), *NU_CLIP)
    s    = np.sqrt((nu - 2.0) / nu)
    q    = tdist.ppf(alpha, df=nu)           # 負值
    var_ret = np.log(np.clip(var_price, 1e-3, None) /
                     np.clip(close_prev, 1e-3, None))  # 負值
    sigma = np.abs(var_ret) / np.abs(s * q)
    return sigma

# 訓練期隱含波動
sigma_implied_tr = implied_sigma(lstm_var_price_tr, prev_close_tr, shape_tr_arr)

# 測試期（WF B版）隱含波動
sigma_implied_wf = implied_sigma(lstm_var_price_wf, prev_close_wf, walk_shapes)

# 測試期（Frozen A版）隱含波動
sigma_implied_frz = implied_sigma(lstm_var_price_frz, prev_close_frz, shape_te)

print(f"\n[Implied Sigma] Train mean={np.nanmean(sigma_implied_tr):.6f}  "
      f"WF mean={np.nanmean(sigma_implied_wf):.6f}  "
      f"GK mean={np.nanmean(walk_gk_vol):.6f}  "
      f"GARCH mean={np.nanmean(walk_garch):.6f}")


# =============================================================================
# 8.  VaR 回測 — Price-Level Violation
#     violation: ES1_CLOSE_t < predicted_VaR_price_t
# =============================================================================

def kupiec_test(viol, alpha=ALPHA_95):
    v = np.asarray(viol, dtype=int)
    n = len(v); x = int(v.sum()); eps = 1e-12
    phat = np.clip(x/n, eps, 1-eps); ac = np.clip(alpha, eps, 1-eps)
    LR_uc = -2*((n-x)*np.log(1-ac)+x*np.log(ac)-(n-x)*np.log(1-phat)-x*np.log(phat))
    return dict(alpha=alpha, n=n, x=x, viol_rate=float(x/n),
                LR_uc=float(LR_uc), p_uc=float(1-chi2.cdf(LR_uc, 1)))

def christoffersen_cc(viol, alpha=ALPHA_95):
    v = np.asarray(viol, dtype=int)
    uc = kupiec_test(v, alpha)
    vl, vn = v[:-1], v[1:]
    n00 = int(np.sum((vl==0)&(vn==0))); n01 = int(np.sum((vl==0)&(vn==1)))
    n10 = int(np.sum((vl==1)&(vn==0))); n11 = int(np.sum((vl==1)&(vn==1)))
    eps = 1e-12
    pi01 = np.clip(n01/max(n00+n01,1), eps, 1-eps)
    pi11 = np.clip(n11/max(n10+n11,1), eps, 1-eps)
    pi   = np.clip((n01+n11)/max(n00+n01+n10+n11,1), eps, 1-eps)
    LR_ind = -2*((n00+n10)*np.log(1-pi)+(n01+n11)*np.log(pi)
                 -n00*np.log(1-pi01)-n01*np.log(pi01)
                 -n10*np.log(1-pi11)-n11*np.log(pi11))
    LR_cc = uc['LR_uc'] + LR_ind
    return dict(**uc, n00=n00, n01=n01, n10=n10, n11=n11,
                LR_ind=float(LR_ind), p_ind=float(1-chi2.cdf(LR_ind,1)),
                LR_cc=float(LR_cc),   p_cc=float(1-chi2.cdf(LR_cc,2)))

def run_bt(viol, alpha, label):
    k = kupiec_test(viol, alpha); c = christoffersen_cc(viol, alpha)
    print(f"  [{label:40s}] N={k['n']} Viol={k['x']} ({k['viol_rate']*100:.2f}%)  "
          f"UC={'✓' if k['p_uc']>=0.05 else '✗'}(p={k['p_uc']:.4f})  "
          f"CC={'✓' if c['p_cc']>=0.05 else '✗'}(p={c['p_cc']:.4f})  "
          f"IND p={c['p_ind']:.4f}  "
          f"n00/01/10/11={c['n00']}/{c['n01']}/{c['n10']}/{c['n11']}")
    return k, c

# 計算 violations（price level）
# 訓練期
viol_lstm_tr  = (close_tr  < lstm_var_price_tr).astype(int)
viol_garch_tr = (ret_tr    < build_var_t_ret(garch_vol_tr_arr, shape_tr_arr)).astype(int)  # ret-level GARCh
viol_gk_tr    = (ret_tr    < gk_var_ret_tr).astype(int)   # GK realized ret-level

# [補充] 統一用 price-level violation（與需求書一致）
valid_tr = ~(np.isnan(lstm_var_price_tr) | np.isnan(garch_var_price_tr) | np.isnan(gk_var_price_tr))
viol_lstm_tr_p  = (close_tr[valid_tr]  < lstm_var_price_tr[valid_tr]).astype(int)
viol_garch_tr_p = (close_tr[valid_tr]  < garch_var_price_tr[valid_tr]).astype(int)
viol_gk_tr_p    = (close_tr[valid_tr]  < gk_var_price_tr[valid_tr]).astype(int)

# 測試期
valid_te = ~(np.isnan(lstm_var_price_wf) | np.isnan(garch_var_price_te) | np.isnan(gk_var_price_te))
viol_lstm_wf_p  = (walk_close[valid_te] < lstm_var_price_wf[valid_te]).astype(int)
viol_garch_te_p = (walk_close[valid_te] < garch_var_price_te[valid_te]).astype(int)
viol_gk_te_p    = (walk_close[valid_te] < gk_var_price_te[valid_te]).astype(int)
viol_lstm_frz_p = (close_te[valid_te]   < lstm_var_price_frz[valid_te]).astype(int)

print("\n" + "="*75)
print("VaR(95%) Backtest — Price-Level Violations")
print("="*75)
print("  [TRAIN 2006-2021]")
k_gk_tr, c_gk_tr     = run_bt(viol_gk_tr_p,    ALPHA_95, "GK Realized (benchmark)")
k_ga_tr, c_ga_tr     = run_bt(viol_garch_tr_p,  ALPHA_95, "GARCH baseline")
k_ls_tr, c_ls_tr     = run_bt(viol_lstm_tr_p,   ALPHA_95, "LSTM_GK Base Model")

print("  [TEST  2022-2025]")
k_gk_te, c_gk_te     = run_bt(viol_gk_te_p,    ALPHA_95, "GK Realized (benchmark)")
k_ga_te, c_ga_te     = run_bt(viol_garch_te_p,  ALPHA_95, "GARCH baseline")
k_ls_frz, c_ls_frz   = run_bt(viol_lstm_frz_p,  ALPHA_95, "LSTM_GK Frozen A-ver")
k_ls_wf,  c_ls_wf    = run_bt(viol_lstm_wf_p,   ALPHA_95, "LSTM_GK Walk-Fwd B-ver")


[Frozen] RMSE=1526.69  MAE=1364.86  Corr=0.9924
✓ Base model saved.

STAGE 2: Walk-Forward Warm Update (2022-2025)
  Warm LR=0.0001, Epochs=1, Batch=1
✓ Warm model weights copied.
Walk-forward: 877 days
  Walk-forward:  200/877 done
  Walk-forward:  400/877 done
  Walk-forward:  600/877 done
  Walk-forward:  800/877 done
  Walk-forward:  877/877 done

[Walk-Fwd B] RMSE=398.79  MAE=339.40  Corr=0.9326
[Frozen   A] RMSE=1526.69  MAE=1364.86  Corr=0.9924
✓ Warm model saved.

[Implied Sigma] Train mean=0.317406  WF mean=0.053730  GK mean=0.009731  GARCH mean=0.010947

VaR(95%) Backtest — Price-Level Violations
  [TRAIN 2006-2021]
  [GK Realized (benchmark)                 ] N=4025 Viol=175 (4.35%)  UC=✓(p=0.0524)  CC=✓(p=0.1248)  IND p=0.5274  n00/01/10/11=3680/169/169/6
  [GARCH baseline                          ] N=4025 Viol=233 (5.79%)  UC=✗(p=0.0249)  CC=✓(p=0.0800)  IND p=0.8838  n00/01/10/11=3572/219/219/14
  [LSTM_GK Base Model                      ] N=4025 Viol=3599 (89.42%)  UC=✗(

In [22]:
# =============================================================================
# 9.  CSV 輸出（需求書第十三節）
# =============================================================================

def make_row(k, c, split, model, level='95'):
    return {'model':model,'split':split,'level':level,
            'N':k['n'],'violations':k['x'],'viol_rate':round(k['viol_rate'],6),
            'kupiec_LR':round(k['LR_uc'],4),'kupiec_p':round(k['p_uc'],4),
            'cc_LR':round(c['LR_cc'],4),'cc_p':round(c['p_cc'],4),
            'ind_p':round(c['p_ind'],4),
            'n00':c['n00'],'n01':c['n01'],'n10':c['n10'],'n11':c['n11'],
            'uc_pass':k['p_uc']>=0.05,'cc_pass':c['p_cc']>=0.05}

# 回測彙總
df_bt = pd.DataFrame([
    make_row(k_gk_tr, c_gk_tr, 'train', 'GK_Realized'),
    make_row(k_ga_tr, c_ga_tr, 'train', 'GARCH'),
    make_row(k_ls_tr, c_ls_tr, 'train', 'LSTM_GK_Base'),
    make_row(k_gk_te, c_gk_te, 'test',  'GK_Realized'),
    make_row(k_ga_te, c_ga_te, 'test',  'GARCH'),
    make_row(k_ls_frz,c_ls_frz,'test',  'LSTM_GK_Frozen'),
    make_row(k_ls_wf, c_ls_wf, 'test',  'LSTM_GK_WalkFwd'),
])
df_bt.to_csv(os.path.join(OUTPUT_DIR,"backtest_summary.csv"), index=False, encoding='utf-8-sig')
print("\n✓ Saved backtest_summary.csv")

# 波動預測誤差（VaR_price 尺度）
df_vm = pd.DataFrame([
    {'model':'LSTM_GK_Train', 'split':'train', 'RMSE':rmse_tr,     'MAE':mae_tr,     'Corr':corr_tr},
    {'model':'LSTM_GK_Frozen','split':'frozen','RMSE':rmse_frozen,  'MAE':mae_frozen,  'Corr':corr_frozen},
    {'model':'LSTM_GK_WalkFwd','split':'wf',   'RMSE':rmse_wf,      'MAE':mae_wf,      'Corr':corr_wf},
])
df_vm.to_csv(os.path.join(OUTPUT_DIR,"forecast_metrics.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved forecast_metrics.csv")

# 三模型 VaR_price 描述統計（需求書：三模型統計比較表）
def desc_arr(arr, name):
    a = pd.Series(arr).dropna()
    return {'name':name,'mean':a.mean(),'median':a.median(),'std':a.std(),'min':a.min(),'max':a.max()}

df_var_stats = pd.DataFrame([
    desc_arr(gk_var_price_tr,   'GK_Realized_VaR_price_train'),
    desc_arr(garch_var_price_tr,'GARCH_VaR_price_train'),
    desc_arr(lstm_var_price_tr, 'LSTM_GK_VaR_price_train'),
    desc_arr(gk_var_price_te,   'GK_Realized_VaR_price_test'),
    desc_arr(garch_var_price_te,'GARCH_VaR_price_test'),
    desc_arr(lstm_var_price_wf, 'LSTM_GK_WF_VaR_price_test'),
])
df_var_stats.to_csv(os.path.join(OUTPUT_DIR,"var_price_stats.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved var_price_stats.csv")

# 隱含波動統計表
df_sig_stats = pd.DataFrame([
    desc_arr(sigma_implied_tr, 'LSTM_implied_sigma_train'),
    desc_arr(gk_vol_tr_arr,    'GK_vol_train'),
    desc_arr(garch_vol_tr_arr, 'GARCH_vol_train'),
    desc_arr(sigma_implied_wf, 'LSTM_implied_sigma_test'),
    desc_arr(walk_gk_vol,      'GK_vol_test'),
    desc_arr(walk_garch,       'GARCH_vol_test'),
])
df_sig_stats.to_csv(os.path.join(OUTPUT_DIR,"implied_sigma_stats.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved implied_sigma_stats.csv")

# 逐日 CSV（訓練期）
df_daily_tr = pd.DataFrame({
    'DATE':              dates_tr,
    'ES1_CLOSE':         close_tr,
    'ES1_LN_RET':        ret_tr,
    'gk_VaR_price_95':   gk_var_price_tr,
    'garch_VaR_price_95':garch_var_price_tr,
    'lstm_VaR_price_95': lstm_var_price_tr,
    'sigma_implied':     sigma_implied_tr,
    'gk_vol_daily':      gk_vol_tr_arr,
    'garch_vol':         garch_vol_tr_arr,
    'viol_gk':           np.where(valid_tr, viol_gk_tr_p, np.nan),
    'viol_garch':        np.where(valid_tr, viol_garch_tr_p, np.nan),
    'viol_lstm':         np.where(valid_tr, viol_lstm_tr_p, np.nan),
}).set_index('DATE')
df_daily_tr.to_csv(os.path.join(OUTPUT_DIR,"daily_train.csv"), encoding='utf-8-sig')
print("✓ Saved daily_train.csv")

# 逐日 CSV（測試期）
df_daily_te = pd.DataFrame({
    'DATE':              walk_dates,
    'ES1_CLOSE':         walk_close,
    'ES1_LN_RET':        walk_ret,
    'gk_VaR_price_95':   gk_var_price_te,
    'garch_VaR_price_95':garch_var_price_te,
    'lstm_VaR_price_wf': lstm_var_price_wf,
    'lstm_VaR_price_frz':lstm_var_price_frz,
    'sigma_implied_wf':  sigma_implied_wf,
    'sigma_implied_frz': sigma_implied_frz,
    'gk_vol_daily':      walk_gk_vol,
    'garch_vol':         walk_garch,
    'viol_gk':           np.where(valid_te, viol_gk_te_p, np.nan),
    'viol_garch':        np.where(valid_te, viol_garch_te_p, np.nan),
    'viol_lstm_wf':      np.where(valid_te, viol_lstm_wf_p, np.nan),
    'viol_lstm_frz':     np.where(valid_te, viol_lstm_frz_p, np.nan),
}).set_index('DATE')
df_daily_te.to_csv(os.path.join(OUTPUT_DIR,"daily_test.csv"), encoding='utf-8-sig')
print("✓ Saved daily_test.csv")


# =============================================================================
# 10.  圖表輸出
# =============================================================================

# ── Fig 1: 訓練損失曲線 ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.history['loss'],     color='#1565C0', lw=1.5, label='Train Loss')
ax.plot(history.history['val_loss'], color='#E53935', lw=1.5, ls='--', label='Val Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Huber Loss')
ax.set_title('Base Model Training Loss (EarlyStopping + ReduceLROnPlateau)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig1_training_loss.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig1_training_loss.png")


# ── Fig 2: 主目標變數預測圖（訓練 & 測試）────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('LSTM_GK Direct VaR Price Prediction vs Actual GK VaR Price',
             fontsize=13, fontweight='bold')

for ax, dates, pred, true, label, rmse, mae, corr in [
    (axes[0], dates_tr, lstm_var_price_tr, gk_var_price_tr,
     f'Train (2006-2021)\nRMSE={rmse_tr:.1f}  MAE={mae_tr:.1f}  Corr={corr_tr:.3f}',
     rmse_tr, mae_tr, corr_tr),
    (axes[1], walk_dates, lstm_var_price_wf, gk_var_price_te,
     f'Walk-Fwd (2022-2025)\nRMSE={rmse_wf:.1f}  MAE={mae_wf:.1f}  Corr={corr_wf:.3f}',
     rmse_wf, mae_wf, corr_wf),
]:
    ax.plot(dates, true, color='#333', lw=0.7, alpha=0.9, label='GK VaR Price (actual)')
    ax.plot(dates, pred, color='#1565C0', lw=0.8, alpha=0.85, label='LSTM_GK Predicted')
    ax.set_title(label, fontsize=10); ax.set_ylabel('VaR Price (Index Points)')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig2_var_price_prediction.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig2_var_price_prediction.png")


# ── Fig 3: 三模型 VaR_price 路徑比較圖（核心圖，需求書第十一節）─────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Three-Model VaR(95%) Price Comparison: GK / GARCH / LSTM_GK',
             fontsize=13, fontweight='bold')

for ax, dates, close_arr, gk_v, ga_v, ls_v, label in [
    (axes[0], dates_tr,  close_tr,   gk_var_price_tr,  garch_var_price_tr, lstm_var_price_tr,
     'Train (2006-2021)'),
    (axes[1], walk_dates, walk_close, gk_var_price_te, garch_var_price_te, lstm_var_price_wf,
     'Walk-Forward (2022-2025)'),
]:
    ax.plot(dates, close_arr, color='#888', lw=0.6, alpha=0.6, label='ES1 Close Price')
    ax.plot(dates, gk_v,  color='#2E7D32', lw=1.2, label='GK VaR95 (benchmark)')
    ax.plot(dates, ga_v,  color='#FF8F00', lw=1.1, ls='--', label='GARCH VaR95')
    ax.plot(dates, ls_v,  color='#1565C0', lw=1.1, ls='-.', label='LSTM_GK VaR95')
    ax.set_title(label, fontsize=11); ax.set_ylabel('Price (Index Points)')
    ax.legend(fontsize=9); ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig3_three_model_var_path.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig3_three_model_var_path.png")


# ── Fig 4: 違規事件圖（三模型，測試期）──────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle('VaR(95%) Price Violations — Test Period 2022-2025\n(GK / GARCH / LSTM_GK Walk-Fwd)',
             fontsize=13, fontweight='bold')

dates_te_valid = walk_dates[valid_te]
for ax, viol_arr, var_arr, label, color in [
    (axes[0], viol_gk_te_p,    gk_var_price_te[valid_te],    'GK Realized',         '#2E7D32'),
    (axes[1], viol_garch_te_p, garch_var_price_te[valid_te], 'GARCH Baseline',      '#FF8F00'),
    (axes[2], viol_lstm_wf_p,  lstm_var_price_wf[valid_te],  'LSTM_GK Walk-Forward','#1565C0'),
]:
    ax.plot(dates_te_valid, walk_close[valid_te], color='#888', lw=0.6, alpha=0.7, label='ES1 Close')
    ax.plot(dates_te_valid, var_arr, color=color, lw=1.0, label=f'{label} VaR95')
    vm = viol_arr == 1
    ax.scatter(dates_te_valid[vm], walk_close[valid_te][vm],
               s=15, color='#D32F2F', zorder=5,
               label=f'Violation (n={viol_arr.sum()}, {viol_arr.mean()*100:.1f}%)')
    ax.set_ylabel('Price'); ax.legend(fontsize=8, loc='lower left'); ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
axes[-1].set_xlabel('Date')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig4_violations_test.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig4_violations_test.png")


# ── Fig 5: 隱含波動比較圖（LSTM implied vs GK vs GARCH）────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Implied Sigma from LSTM VaR Price vs GK / GARCH Volatility',
             fontsize=13, fontweight='bold')

for ax, dates, sig_lstm, sig_gk, sig_garch, label in [
    (axes[0], dates_tr,   sigma_implied_tr, gk_vol_tr_arr,    garch_vol_tr_arr, 'Train (2006-2021)'),
    (axes[1], walk_dates, sigma_implied_wf, walk_gk_vol,      walk_garch,       'Walk-Fwd (2022-2025)'),
]:
    ax.plot(dates, sig_gk,    color='#2E7D32', lw=0.8, alpha=0.85, label='GK vol (actual)')
    ax.plot(dates, sig_garch, color='#FF8F00', lw=0.8, alpha=0.85, ls='--', label='GARCH vol')
    ax.plot(dates, sig_lstm,  color='#1565C0', lw=0.9, alpha=0.9,  ls='-.', label='LSTM implied σ')
    ax.set_title(label, fontsize=11); ax.set_ylabel('Volatility σ')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig5_implied_sigma.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig5_implied_sigma.png")


# ── Fig 6: 誤差診斷圖（Scatter + Quintile Bias + Std Ratio）────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Prediction Diagnosis: VaR Price Level\n(Scatter / Quintile Bias / Variance Ratio)',
             fontsize=12, fontweight='bold')

# Scatter
ax = axes[0]
ax.scatter(gk_var_price_te, lstm_var_price_wf, s=5, alpha=0.3, color='#1565C0', label='Walk-Fwd')
ax.scatter(gk_var_price_te, lstm_var_price_frz, s=5, alpha=0.25, color='#E53935', label='Frozen')
lim = max(np.nanmax(gk_var_price_te), np.nanmax(lstm_var_price_wf)) * 1.05
ax.plot([0,lim],[0,lim],'k-',lw=1.2,label='45° (perfect)')
ax.set_xlabel('GK VaR Price (actual)'); ax.set_ylabel('LSTM VaR Price (pred)')
ax.set_title(f'Scatter\nWF Corr={corr_wf:.3f}, Frz Corr={corr_frozen:.3f}', fontsize=9)
ax.legend(fontsize=8); ax.grid(alpha=0.25)

# Quintile bias
ax = axes[1]
valid_both = ~(np.isnan(gk_var_price_te) | np.isnan(lstm_var_price_wf) | np.isnan(lstm_var_price_frz))
gk_v = gk_var_price_te[valid_both]
ls_v = lstm_var_price_wf[valid_both]
fr_v = lstm_var_price_frz[valid_both]
try:
    q = pd.qcut(gk_v, 5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
    qnames = ['Q1','Q2','Q3','Q4','Q5']
    bias_wf  = [(ls_v[q==qn]-gk_v[q==qn]).mean() for qn in qnames]
    bias_frz = [(fr_v[q==qn]-gk_v[q==qn]).mean() for qn in qnames]
    x5 = np.arange(5)
    ax.bar(x5-0.2, bias_wf,  0.35, color='#1565C0', label='Walk-Fwd', alpha=0.8)
    ax.bar(x5+0.2, bias_frz, 0.35, color='#E53935', label='Frozen',   alpha=0.8)
    ax.axhline(0, color='k', lw=1, ls='--')
    ax.set_xticks(x5); ax.set_xticklabels(['Q1\n(low)','Q2','Q3','Q4','Q5\n(high)'])
except Exception:
    ax.text(0.5, 0.5, 'Quintile unavailable', ha='center', transform=ax.transAxes)
ax.set_ylabel('Mean Bias (Pred - GK)')
ax.set_title('Bias by GK-VaR Quintile\n(negative=underestimate risk)', fontsize=9)
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Variance suppression
ax = axes[2]
std_ratios = [
    np.nanstd(lstm_var_price_tr)  / np.nanstd(gk_var_price_tr),
    np.nanstd(lstm_var_price_frz) / np.nanstd(gk_var_price_te),
    np.nanstd(lstm_var_price_wf)  / np.nanstd(gk_var_price_te),
]
labs = ['Train', 'Frozen', 'Walk-Fwd']
bars = ax.bar(labs, std_ratios, color=['#1565C0','#E53935','#7B1FA2'], edgecolor='white', width=0.45)
ax.axhline(1.0, color='k', lw=1.5, ls='--', label='Perfect = 1.0')
for bar, v in zip(bars, std_ratios):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Pred Std / GK Std')
ax.set_title('Variance Ratio\n(<1 = model compresses VaR range)', fontsize=9)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 1.4)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig6_diagnosis.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig6_diagnosis.png")


# ── Fig 7: VaR Violation Rate 比較柱狀圖 ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VaR(95%) Violation Rate — Price Level\n(GK / GARCH / LSTM_GK)',
             fontsize=13, fontweight='bold')

for ax, rows, title in [
    (axes[0],
     [('GK\nBenchmark', k_gk_tr['viol_rate'], '#2E7D32'),
      ('GARCH\nBaseline', k_ga_tr['viol_rate'], '#FF8F00'),
      ('LSTM_GK\nBase', k_ls_tr['viol_rate'],  '#1565C0')],
     'Train (2006-2021)'),
    (axes[1],
     [('GK\nBenchmark', k_gk_te['viol_rate'], '#2E7D32'),
      ('GARCH\nBaseline', k_ga_te['viol_rate'], '#FF8F00'),
      ('LSTM Frozen\n(A-ver)', k_ls_frz['viol_rate'],'#E53935'),
      ('LSTM WF\n(B-ver)', k_ls_wf['viol_rate'], '#7B1FA2')],
     'Test (2022-2025)'),
]:
    names = [r[0] for r in rows]; vrs = [r[1]*100 for r in rows]; cols = [r[2] for r in rows]
    bars = ax.bar(names, vrs, color=cols, edgecolor='white', width=0.5)
    ax.axhline(5.0, color='navy', ls='--', lw=1.5, label='5% target')
    for bar, v in zip(bars, vrs):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.15,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylabel('Violation Rate (%)'); ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max(vrs)*1.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig7_violation_rate.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig7_violation_rate.png")


# ── Fig 8: Annual violation rate（Walk-Fwd 測試期）────────────────────────────
ann_gk  = (pd.DataFrame({'year':walk_dates.year,'v':viol_gk_te_p.tolist()+[np.nan]*(len(walk_dates)-len(viol_gk_te_p))})
           .groupby('year').agg(vr=('v','mean')).reset_index())

dates_vte = walk_dates[valid_te]
ann = pd.DataFrame({
    'year': dates_vte.year,
    'gk':   viol_gk_te_p, 'ga': viol_garch_te_p, 'ls': viol_lstm_wf_p
}).groupby('year').mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle('Annual VaR(95%) Violation Rate — Test Period 2022-2025', fontsize=12, fontweight='bold')
x_pos = np.arange(len(ann))
w = 0.25
ax.bar(x_pos-w,   ann['gk']*100, w, color='#2E7D32', label='GK')
ax.bar(x_pos,     ann['ga']*100, w, color='#FF8F00', label='GARCH')
ax.bar(x_pos+w,   ann['ls']*100, w, color='#1565C0', label='LSTM_GK WF')
ax.axhline(5.0, color='navy', ls='--', lw=1.5, label='5% target')
ax.set_xticks(x_pos); ax.set_xticklabels(ann['year'].astype(str))
ax.set_ylabel('Violation Rate (%)'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig8_annual_violation.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig8_annual_violation.png")


# ── Fig 9: Rolling 60-day violation (測試期) ────────────────────────────────
df_roll = pd.DataFrame({
    'gk': viol_gk_te_p, 'ga': viol_garch_te_p, 'ls': viol_lstm_wf_p
}, index=dates_te_valid)

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(dates_te_valid, df_roll['gk'].rolling(60).mean(), color='#2E7D32', lw=1.5, label='GK')
ax.plot(dates_te_valid, df_roll['ga'].rolling(60).mean(), color='#FF8F00', lw=1.5, ls='--', label='GARCH')
ax.plot(dates_te_valid, df_roll['ls'].rolling(60).mean(), color='#1565C0', lw=1.5, ls='-.', label='LSTM_GK WF')
ax.axhline(0.05, color='navy', ls=':', lw=1.2, label='5% target')
ax.set_ylabel('Rolling 60d Violation Rate')
ax.set_title('Rolling 60-Day VaR(95%) Violation Rate — Test 2022-2025')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30); ax.legend(fontsize=9); ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,"fig9_rolling_violation.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ fig9_rolling_violation.png")


# =============================================================================
# 11.  最終摘要
# =============================================================================

print("\n" + "="*75)
print("FINAL SUMMARY")
print("="*75)
print(f"\n[VaR Price Forecast — MAE / RMSE / Corr]")
print(f"  Train (Base):   MAE={mae_tr:.2f}  RMSE={rmse_tr:.2f}  Corr={corr_tr:.4f}")
print(f"  Frozen A-ver:   MAE={mae_frozen:.2f}  RMSE={rmse_frozen:.2f}  Corr={corr_frozen:.4f}")
print(f"  Walk-Fwd B-ver: MAE={mae_wf:.2f}  RMSE={rmse_wf:.2f}  Corr={corr_wf:.4f}")

print(f"\n[Implied Sigma vs GK / GARCH — Test Period]")
print(f"  LSTM implied σ: mean={np.nanmean(sigma_implied_wf):.6f}")
print(f"  GK vol daily:   mean={np.nanmean(walk_gk_vol):.6f}")
print(f"  GARCH vol:      mean={np.nanmean(walk_garch):.6f}")

print(f"\n[VaR(95%) Backtest — Price Level]")
for name, k, c in [
    ("GK Realized  (Train)", k_gk_tr, c_gk_tr),
    ("GARCH        (Train)", k_ga_tr, c_ga_tr),
    ("LSTM_GK Base (Train)", k_ls_tr, c_ls_tr),
    ("GK Realized  (Test) ", k_gk_te, c_gk_te),
    ("GARCH        (Test) ", k_ga_te, c_ga_te),
    ("LSTM_GK Frozen (Test)", k_ls_frz, c_ls_frz),
    ("LSTM_GK WalkFwd(Test)", k_ls_wf,  c_ls_wf),
]:
    print(f"  {name:<28s}: {k['viol_rate']*100:.2f}%  "
          f"UC={'PASS' if k['p_uc']>=0.05 else 'FAIL'}(p={k['p_uc']:.4f})  "
          f"CC={'PASS' if c['p_cc']>=0.05 else 'FAIL'}(p={c['p_cc']:.4f})")

print(f"\n✓ All outputs: {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f"  {f:<45s} {os.path.getsize(fp)//1024:5d} KB")


✓ Saved backtest_summary.csv
✓ Saved forecast_metrics.csv
✓ Saved var_price_stats.csv
✓ Saved implied_sigma_stats.csv
✓ Saved daily_train.csv
✓ Saved daily_test.csv
✓ fig1_training_loss.png
✓ fig2_var_price_prediction.png
✓ fig3_three_model_var_path.png
✓ fig4_violations_test.png
✓ fig5_implied_sigma.png
✓ fig6_diagnosis.png
✓ fig7_violation_rate.png
✓ fig8_annual_violation.png
✓ fig9_rolling_violation.png

FINAL SUMMARY

[VaR Price Forecast — MAE / RMSE / Corr]
  Train (Base):   MAE=1085.39  RMSE=1192.10  Corr=0.9742
  Frozen A-ver:   MAE=1364.86  RMSE=1526.69  Corr=0.9924
  Walk-Fwd B-ver: MAE=339.40  RMSE=398.79  Corr=0.9326

[Implied Sigma vs GK / GARCH — Test Period]
  LSTM implied σ: mean=0.053730
  GK vol daily:   mean=0.009731
  GARCH vol:      mean=0.010947

[VaR(95%) Backtest — Price Level]
  GK Realized  (Train)        : 4.35%  UC=PASS(p=0.0524)  CC=PASS(p=0.1248)
  GARCH        (Train)        : 5.79%  UC=FAIL(p=0.0249)  CC=PASS(p=0.0800)
  LSTM_GK Base (Train)        : 89.